In [10]:
import os
import glob

from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_openai import ChatOpenAI

import gradio as gr

load_dotenv(override=True)

True

In [3]:
from langchain_community.document_loaders import JSONLoader

loader = JSONLoader(
    file_path="corpus.jsonl",
    # jq_schema="._id",
    jq_schema=".text",
    json_lines=True
)

documents = loader.load()
print(f"Loaded {len(documents)} documents")

Loaded 973 documents


In [4]:
print(type(documents[0]))
documents[0].page_content[:500]  # Print the first 500 characters of the first document

<class 'langchain_core.documents.base.Document'>


'"Privileged" Nominations Every year the Senate routinely considers whether to give its advice and consent to hundreds of nominations submitted by the President. From start to finish, the confirmation process can be a lengthy one, even for relatively noncontroversial nominees. Each nomination is typically referred to one or more committees having subject matter jurisdiction over the position. Committees may bear a significant workload in examining nomineesâ\x80\x94often including questionnaires, option'

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

textsplitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)

chunks = textsplitter.split_documents(documents)
print(f"Split into {len(chunks)} chunks")

Split into 109064 chunks


In [6]:
huge_file = "corpus.jsonl"
small_file = "small_corpus.jsonl"
lines_to_keep = 50  # 50 documents is perfect for a quick test

print(f"Extracting first {lines_to_keep} lines from {huge_file}...")
with open(huge_file, "r", encoding="utf-8") as infile, \
     open(small_file, "w", encoding="utf-8") as outfile:
    for i, line in enumerate(infile):
        if i >= lines_to_keep:
            break
        outfile.write(line)
print(f"Created {small_file}")

Extracting first 50 lines from corpus.jsonl...
Created small_corpus.jsonl


In [7]:
loader = JSONLoader(
    file_path=small_file,
    jq_schema=".text",
    json_lines=True
)
documents = loader.load()
print(f"Loaded {len(documents)} documents from small corpus")

textsplitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = textsplitter.split_documents(documents)
print(f"Split into {len(chunks)} chunks")

Loaded 50 documents from small corpus
Split into 5605 chunks


In [ ]:
# ====================================================================
# 3. GENERATE EMBEDDINGS (CPU OPTIMIZED)
# ====================================================================
print("Initializing CPU-friendly embedding model...")
# Using 'bge-small' instead of 'bge-large' - it is much faster on CPU!
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5", 
    model_kwargs={'device': 'cpu'}, 
    encode_kwargs={'normalize_embeddings': True}
)

Initializing CPU-friendly embedding model...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1081.26it/s]


In [ ]:
# ====================================================================
# 4. PUSH TO PINECONE
# ====================================================================
from langchain_pinecone import PineconeVectorStore
from pinecone import Pinecone
from pinecone import ServerlessSpec

# Make sure PINECONE_API_KEY is in your .env file
pinecone_api_key = os.getenv("PINECONE_API_KEY")
pc = Pinecone(api_key=pinecone_api_key)

# Use a new index name since the dimension changed from 1024 to 384
index_name = "gov-docs-index-small" 

# Automatically create the index if it doesn't exist
if not pc.has_index(index_name):
    print(f"Creating new Pinecone index '{index_name}' with dimension 384...")
    pc.create_index(
        name=index_name,
        dimension=384, # bge-small uses 384 dimensions
        metric="cosine", # Cosine is recommended for BGE models
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )

print(f"Pushing {len(chunks)} chunks to Pinecone index: '{index_name}'...")
vectorstore = PineconeVectorStore.from_documents(
    documents=chunks,
    embedding=embeddings,
    index_name=index_name
)

print("✅ Embeddings generated and stored in Pinecone successfully!")


Creating new Pinecone index 'gov-docs-index-small' with dimension 384...
Pushing 5605 chunks to Pinecone index: 'gov-docs-index-small'...
✅ Embeddings generated and stored in Pinecone successfully!


In [15]:
# Test the retriever quickly
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 5})
test_docs = retriever.invoke("What is the defense acquisition system?")
print(f"\nTest retrieval found {len(test_docs)} documents.")


Test retrieval found 5 documents.


In [ ]:
from langchain_openrouter import ChatOpenRouter
llm = ChatOpenRouter(model="nousresearch/hermes-3-llama-3.1-405b:free")

In [17]:
retriever.invoke("What is Defense Acquisition System?")

[Document(id='3008395a-eb9e-41d7-83f3-a6ee66313662', metadata={'seq_num': 4.0, 'source': '/home/due_dilligence/small_corpus.jsonl'}, page_content='Background DOD acquires new weapon systems for its warfighters through a management process known as the Defense Acquisition System. This system is implemented by two key acquisition policies: DOD Directive 5000.01, which establishes the overarching framework for the Defense Acquisition System; and DOD Instruction 5000.02, which provides detailed procedures for the operation of the Defense Acquisition System and the management of acquisition programs. These policy documents establish the guiding'),
 Document(id='90de1087-af3f-431e-91c5-5b683b764390', metadata={'seq_num': 13.0, 'source': '/home/due_dilligence/small_corpus.jsonl'}, page_content='to an approved need. DOD Directive 5000.01, The Defense Acquisition System, provides management principles and mandatory policies and procedures for managing all acquisition programs. Oversight levels 

In [21]:
llm.invoke("What is Defense Acquisition System?")

AIMessage(content='The **Defense Acquisition System (DAS)** is the **overarching framework of policies, processes, regulations, and organizational structures** used by the **United States Department of Defense (DoD)** to acquire weapons systems, equipment, services, and information technology needed to support national defense objectives. It governs *how* the DoD identifies needs, develops requirements, selects contractors, manages development and production, fields systems, and provides logistics support throughout a system\'s lifecycle.\n\nThink of it as the DoD\'s "rulebook" and "operating manual" for buying everything from fighter jets and ships to software, uniforms, and logistics support.\n\n### Key Foundations & Governing Documents\nThe DAS is primarily established and governed by:\n1.  **Statutory Basis:** Title 10 of the United States Code (U.S.C.), particularly Chapters 131-138 (Acquisition).\n2.  **DoD Directive 5000.01:** *The Defense Acquisition System* (Issued by the Secr

In [20]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, strict assistant representing the details from government documents.
You are chatting with a user about government policies.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

In [22]:
def answer_question_with_reranking(question: str):
    print(f"\n[?] Question: {question}")
    
    # 1. Retrieve & Rerank
    # This fetches 10 from Pinecone, then reranks and compresses down to top 3
    docs = compression_retriever.invoke(question)
    print(f"[*] Retrieved & Reranked {len(docs)} highly relevant documents.")
    
    # 2. Format Context
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    
    # 3. Generate Answer
    response = llm.invoke([
        SystemMessage(content=system_prompt), 
        HumanMessage(content=question)
    ])
    
    print("\n[+] Answer:")
    print(response.content)
    return response.content

In [25]:
if __name__ == "__main__":
    answer_question_with_reranking("What is the defense acquisition system?")


[?] Question: What is the defense acquisition system?


NameError: name 'compression_retriever' is not defined